# Coding, scaling, and the linear model

**Source worksheet:** [yint.org/w2](https://yint.org/w2) - week 2 of the applied DoE short course.

Module 1 read effects straight off corner averages of a 2x2. That was
appropriate for two factors and four runs, but the same approach
becomes cumbersome
once you scale beyond it. Module 2 introduces the *coded linear model*
- the workhorse that scales to any number of factors, copes with
interactions, and gives you predictions outside the corners you ran.

The math is unchanged from Module 1; what we add is **coded units**
(-1 and +1 instead of grams and minutes), a **linear regression** that
recovers the same effects you computed by hand, and a sober look at
**extrapolation** - what the model promises, and what it cannot promise.

Everything below uses Python with `process_improve`; the original
worksheet's R snippet at ``yint.org/2w1`` is translated to its Python
equivalent.

## Questions 1 and 2 - how many runs does a full factorial need?

A full factorial runs every combination of every factor level. For *k*
factors at *L* levels each, that is **L**^**k** experiments.

1. Four factors, each at two levels: how many runs?
2. Four factors, each at three levels: how many runs?

In [ ]:
def full_factorial_size(num_factors: int, levels: int) -> int:
    return levels ** num_factors


two_level = full_factorial_size(num_factors=4, levels=2)
three_level = full_factorial_size(num_factors=4, levels=3)
print(f"Full factorial, 4 factors x 2 levels: {two_level} runs")
print(f"Full factorial, 4 factors x 3 levels: {three_level} runs")

## Questions 3 to 9 - greenhouse plant heights

A greenhouse experiment varies two factors to maximize plant height:

- **A = soil amount** [200 g or 400 g]
- **B = water amount** [50 mL or 100 mL]

The runs come out of the lab notebook *in experiment order*, not in
the textbook's standard order:

| Run | A (g) | B (mL) | Height (mm) |
|----:|------:|-------:|------------:|
|  1  |  200  |   100  |     61      |
|  2  |  200  |    50  |     39      |
|  3  |  400  |   100  |     82      |
|  4  |  400  |    50  |     50      |

The worksheet asks for the rewrite in standard order, the main effects
of A and B, the prediction equation, and three predicted heights at
new conditions.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

from process_improve.experiments import c, gather, lm, main_effects_plot, predict
from process_improve.experiments.visualization import visualize_doe

pio.renderers.default = "notebook_connected"

# Build the design directly in standard order: A varies fastest, then B.

A = c(-1, +1, -1, +1, lo=200, hi=400, name="A", units="grams")
B = c(-1, -1, +1, +1, lo=50,  hi=100, name="B", units="mL")
height = c(39, 50, 61, 82, name="height")
plant = gather(A=A, B=B, height=height)
plant

In [ ]:
# Fit a main-effects model in coded units.

model = lm("height ~ A + B", plant)
params = model.get_parameters(drop_intercept=False)
print(params.to_string())

effect_A = 2 * params["A"]
effect_B = 2 * params["B"]
print(f"\nMain effect of soil amount (A):  {effect_A:+.1f} mm")
print(f"Main effect of water amount (B): {effect_B:+.1f} mm")

In [ ]:
# Verify the prediction equation by predicting at three test points.

points = [
    ("300 g soil, 75 mL water", dict(A=0,  B=0)),
    ("200 g soil, 75 mL water", dict(A=-1, B=0)),
    ("500 g soil, 125 mL water (extrapolation!)", dict(A=2, B=2)),
]
for label, coded in points:
    y_hat = float(predict(model, **coded).iloc[0])
    print(f"{label:50s} -> {y_hat:6.1f} mm")

In [ ]:
fig = main_effects_plot(model, factor_labels={"A": "Soil [g]", "B": "Water [mL]"})
fig.update_layout(width=620, height=380, title="Main-effects plot - greenhouse")
fig

## Questions 10 to 12 - extrapolation, one-factor-at-a-time, and interactions

The worksheet now steps back from the maths:

10. Can you extrapolate outside the design region? How far? And what are
    contour lines useful for?
11. A colleague wants to run their next study by changing **one factor
    at a time** against a baseline. Why is that a bad idea?
12. Give an everyday example of an interaction where increasing factor B:

    - *amplifies* the effect of factor A;
    - *cancels* or *reverses* the effect of factor A.

## Question 13 - a baked food with a negative interaction

A product is baked for time *T* using a certain amount of fat *F*. The
outcome is *crispiness*. Both factors have plausible main effects, but
the data hide a negative interaction that turns up only when you fit
the model.

| Run | T (min) | F (g) | Crispiness |
|----:|--------:|------:|-----------:|
|  1  |    80   |   20  |     37     |
|  2  |   100   |   20  |     57     |
|  3  |    80   |   30  |     49     |
|  4  |   100   |   30  |     53     |

The worksheet asks for the cube plot with contours, an interaction plot
in *both* orientations, a verification of the prediction equation, and
two extrapolated crispness predictions.

In [ ]:
T = c(-1, +1, -1, +1, lo=80, hi=100, name="T", units="min")
F = c(-1, -1, +1, +1, lo=20, hi=30,  name="F", units="g")
crisp = c(37, 57, 49, 53, name="crisp")
bake = gather(T=T, F=F, crisp=crisp)
bake

In [ ]:
square = visualize_doe(
    plot_type="square_plot",
    design_data=bake.to_dict(orient="records"),
    response_column="crisp",
    factors_to_plot=["T", "F"],
    factor_labels={"T": "Time [min]", "F": "Fat [g]"},
    backend="plotly",
)
fig = go.Figure(square["plotly"])
fig.update_layout(width=520, height=440, title="Baked-food 2x2: crispiness")
fig

In [ ]:
model = lm("crisp ~ T + F + T:F", bake)
params = model.get_parameters(drop_intercept=False)
print(params.to_string())
print()
print("Verification - the equation is:")
print(f"  crisp = {params['Intercept']:.0f} "
      f"+ {params['T']:+.0f} * x_T "
      f"+ {params['F']:+.0f} * x_F "
      f"+ {params['T:F']:+.0f} * x_T * x_F")

In [ ]:
# Interaction plot: lines with very different slopes signal interaction.
# visualize_doe(plot_type="interaction") draws both orientations from
# the same call (Plotly subplots) so you can read it from either factor's
# point of view.

interaction = visualize_doe(
    plot_type="interaction",
    design_data=bake.to_dict(orient="records"),
    response_column="crisp",
    factors_to_plot=["T", "F"],
    factor_labels={"T": "Time [min]", "F": "Fat [g]"},
    backend="plotly",
)
fig = go.Figure(interaction["plotly"])
fig.update_layout(width=720, height=380, title="Interaction T x F (both orientations)")
fig

In [ ]:
# Predictions at the two extrapolation points the worksheet asks for.
# In coded units: T-center = 90 min, half-range = 10 min;
#                 F-center = 25 g,  half-range = 5 g.

points = [
    ("110 min, 20 g fat", dict(T=2, F=-1)),
    ("110 min, 15 g fat", dict(T=2, F=-2)),
]
for label, coded in points:
    y_hat = float(predict(model, **coded).iloc[0])
    print(f"{label:25s} -> crispness {y_hat:.1f}")

## Question 14 - the code from the source worksheet

The week-2 worksheet links to an R snippet at
[yint.org/2w1](https://yint.org/2w1) that fits the baked-food
interaction model.  The same job in `process_improve` is the
block above:

```python
from process_improve.experiments import c, gather, lm

T = c(-1, +1, -1, +1)
F = c(-1, -1, +1, +1)
y = c(37, 57, 49, 53)
bake = gather(T=T, F=F, y=y)
model = lm("y ~ T + F + T:F", bake)
print(model.get_parameters(drop_intercept=False))
```

No R required.  The library exposes `lm()`
with the same formula syntax (patsy under the hood) and a
`Model.get_parameters()`
helper that returns the coefficients as a pandas Series.

## Wrap-up

You have now moved from "four numbers and a manual calculation" to a
**coded linear model** that predicts anywhere in the design space, and
beyond it with appropriate caution.  Two notable habits:

- Coding factors to ``[-1, +1]`` makes coefficients directly comparable
  and turns the prediction equation into something you can read off in
  your head.
- An interaction term costs nothing extra to fit in a 2x2 and tells
  you when "more is better" stops being true.

**Next:** Module 3 scales the same toolkit to **three factors** (the
cube plot is back) and replicates the worksheet from week 3, where a
fifth experiment at the center is added to gain a degree of freedom
and start estimating curvature.